# Task 2: Leakage-Free Feature Engineering
## Walmart StockSense — IntelliData 2026 Hackathon

**Prediction Time Assumption:** END OF DAY $t$.
Features represent exclusively historical state realized at or before day $t$. Future observations are strictly reserved for target construction.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
SERVER_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR / 'server'
DATA_PROCESSED = SERVER_DIR / 'data' / 'processed'

from server.src.task2_features import generate_model_features

df_features = generate_model_features()
print(f"\nEngineered Features Dataset: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns")


### Feature Inventory & Warmup Retention
Examine lag, rolling, inventory coverage, trend, pricing, and event features.


In [ ]:
feature_categories = {
    "Demand Lags": ["lag_1", "lag_7", "lag_14"],
    "Rolling Statistics (<= t)": ["rolling_mean_3", "rolling_mean_7", "rolling_mean_14", "rolling_std_3", "rolling_std_7", "rolling_std_14"],
    "Demand Momentum / Growth": ["demand_growth_7d", "demand_growth_14d", "recent_vs_baseline_demand"],
    "Inventory & Buffer Metrics": ["current_inventory", "incoming_inventory", "inventory_coverage_days", "inventory_to_demand_ratio", "reorder_gap", "safety_stock_gap", "lead_time_demand", "recent_stockouts_7d"],
    "Price & Promotion": ["current_price", "price_vs_mrp", "price_change_7d", "discount_pct", "promotion_recent_frequency"],
    "Targets": ["next_7_day_demand", "target_stockout_7d", "supervised_flag"]
}

for cat, cols in feature_categories.items():
    print(f"\n--- {cat} ({len(cols)} columns) ---")
    display(df_features[cols].head(3))


### Supervised vs Prospective Evaluation Split
Verify the chronological availability of ground-truth 7-day forward targets.


In [ ]:
print(f"Total Observations: {len(df_features):,}")
print(f"Supervised Rows (complete 7-day target): {(df_features['supervised_flag'] == 1).sum():,}")
print(f"Prospective Evaluation Rows (unlabeled future test window): {(df_features['supervised_flag'] == 0).sum():,}")
print(f"Future Stockout Positive Rate: {df_features[df_features['supervised_flag'] == 1]['target_stockout_7d'].mean():.2%}")
